# ITSPM: Colab quickstart
Choose **Runtime → Change runtime type → GPU**, then run the setup and small examples.
These examples check the pipeline; they do not reproduce full benchmark scores.
The default data are bundled or generated locally. P12, PAM, USHCN, Stocks and Energy need separate preparation; see the [data guide](https://github.com/zhouziyu02/IP_ITSPM/blob/main/docs/DATASETS.md).


In [1]:
%cd /content
!git clone https://github.com/zhouziyu02/IP_ITSPM.git
%cd /content/IP_ITSPM
%pip install -q -r requirements.txt


/content
Cloning into 'IP_ITSPM'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 133 (delta 11), reused 133 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 20.70 MiB | 28.34 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/IP_ITSPM
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.7 MB/s eta 0:00:00


In [2]:
import torch
print("PyTorch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
!python3 scripts/check_data.py --dataset bundled --verify


PyTorch: 2.11.0+cu130 CUDA available: True
GPU: Tesla T4
OK activity
OK taxi
OK mujoco
VERIFIED data/activity/processed/data.pt
VERIFIED TPP/taxi/train.pkl
VERIFIED TPP/taxi/dev.pkl
VERIFIED TPP/taxi/test.pkl
VERIFIED Generation/itspm_generation/table1_data/mujoco_training_36.pt


## 1. Small real-data forecasting run
Activity is already bundled. This uses eight source records, one epoch and a small model.
Training/validation/test identities are split before window expansion. The full preset is `Extrapolation/scripts/activity.sh`.


In [3]:
!python3 regression.py --model ITSPM --dataset activity --task forecasting --history 3000 -n 8 --epoch 1 --batch_size 8 --d_model 16 --n_ref_points 8 --n_scales 2 --n_mixer_layers 1 --max_event_tokens 16 --max_gap_tokens 8 --seed 0 --state colab_quickstart


PID, device: 775 cuda
/content/IP_ITSPM/regression.py
2026-10-23 06:53:58
regression.py --model ITSPM --dataset activity --task forecasting --history 3000 -n 8 --epoch 1 --batch_size 8 --d_model 16 --n_ref_points 8 --n_scales 2 --n_mixer_layers 1 --max_event_tokens 16 --max_gap_tokens 8 --seed 0 --state colab_quickstart
Namespace(state='debug', model='ITSPM', root_path='', data_path='/content/IP_ITSPM/data', n=8, epoch=1, batch_size=8, num_types=23, d_model=16, dropout=0.0, lr=0.001, gpu='0', log='./logs/', save_path='./checkpoints/regression/', seed=0, patience=10, weight_decay=1e-05, load_path=None, test_only=False, logmode='a', task='forecasting', debug_flag=False, dp_flag=False, load_in_batch=False, history=3000, retrain=False, median_len=50, load=None, dataset='activity', quantization=0.0, max_len=-1, sample_rate=1.0, mask_rate=0.3, collate='indseq', n_ref_points=8, n_scales=2, n_mixer_layers=1, max_event_tokens=16, max_gap_tokens=8, device=device(type='cuda'), PID=775)
Total reco

## 2. Small generation run
Sines needs no external download. Two epochs and ten discriminator steps are only a functionality check, not a useful quality estimate.


In [4]:
!SEQ_LENS=12 MISSING_RATIOS=0.5 N_SAMPLES=256 DS_ITERATIONS=10 GPU=0 SEED=0 bash Generation/scripts/sines.sh --ae_epochs 2 --diff_epochs 2 --batch_size 16


/content/IP_ITSPM/models/ITSPM.py:512: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.token_mixer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
ae_epoch=001 loss=0.938494
diff_epoch=001 loss=1.000238
{
  "gaussian_baseline": {
    "acf_error": 0.010900115594267845,
    "ds": 0.0,
    "kl": 0.03789685250212398,
    "mdd": 3.9755685329437256,
    "mmd": 0.035642750561674985
  },
  "itspm_latent_diffusion": {
    "acf_error": 0.01912151649594307,
    "ds": 0.0,
    "kl": 9.46407224726602,
    "mdd": 6.843851566314697,
    "mmd": 0.31691921774590215
  }
}


## 3. CPU regression checks
This covers all task families, including classification on synthetic fixtures and a real Taxi batch.


In [5]:
!bash scripts/run_checks.sh


OK activity
OK taxi
OK mujoco
VERIFIED data/activity/processed/data.pt
VERIFIED TPP/taxi/train.pkl
VERIFIED TPP/taxi/dev.pkl
VERIFIED TPP/taxi/test.pkl
VERIFIED Generation/itspm_generation/table1_data/mujoco_training_36.pt
test_both_classification_collates (test_core.CoreTests.test_both_classification_collates) ... ok
test_checkpoint_roundtrip (test_core.CoreTests.test_checkpoint_roundtrip) ... ok
ok
test_empty_windows_and_masks (test_core.CoreTests.test_empty_windows_and_masks) ... Excluded 1 records/windows without observed inputs or prediction targets.
Excluded 1 records/windows without observed inputs or prediction targets.
Excluded 1 USHCN windows without observed inputs or prediction targets.
ok
test_error_metrics (test_core.CoreTests.test_error_metrics) ... ok
test_invalid_input_shapes_are_rejected (test_core.CoreTests.test_invalid_input_shapes_are_rejected) ... ok
test_model_forward_and_backward (test_core.CoreTests.test_model_forward_and_backward) ... ok
test_partial_class_met

## Continue with your project
Read [README](https://github.com/zhouziyu02/IP_ITSPM), [project milestones](https://github.com/zhouziyu02/IP_ITSPM/blob/main/docs/PROJECTS.md), and [reference-result limitations](https://github.com/zhouziyu02/IP_ITSPM/blob/main/docs/REFERENCE_RESULTS.md).

After preparing the chosen dataset, run its preset with explicit seeds 0, 1 and 2, keep splits and budgets fixed, and report mean ± sample standard deviation. Trailing flags such as `--epoch 5 --batch_size 8` override classification/regression presets. The generation head still needs a predictive-score evaluator for project 1.5; projects 1.6–1.8 require extensions.

Checkpoints, results and logs are local and ignored by Git. Copy them to your own persistent storage before the Colab runtime disconnects; mounted Google Drive is one option. Do not treat a short run or historical single-seed result as a completed reproduction.


In [6]:
%cd /content/IP_ITSPM

!GPU=0 SEED=0 bash Extrapolation/scripts/activity.sh --epoch 5

/content/IP_ITSPM
PID, device: 5738 cuda
/content/IP_ITSPM/regression.py
2026-10-23 07:13:41
regression.py --model ITSPM --gpu 0 --dataset activity --task forecasting --seed 0 --epoch 1000 --patience 10 --batch_size 32 --d_model 128 --dropout 0.05 --lr 5e-4 --weight_decay 1e-5 --n_ref_points 64 --n_scales 4 --n_mixer_layers 3 --history 3000 --state itspm_extrapolation_activity_best_seed0_20261023_071338 --epoch 5
Namespace(state='itspm_extrapolation_activity_best_seed0_20261023_071338', model='ITSPM', root_path='', data_path='/content/IP_ITSPM/data', n=100000000, epoch=5, batch_size=32, num_types=23, d_model=128, dropout=0.05, lr=0.0005, gpu='0', log='./logs/', save_path='./checkpoints/regression/', seed=0, patience=10, weight_decay=1e-05, load_path=None, test_only=False, logmode='a', task='forecasting', debug_flag=False, dp_flag=False, load_in_batch=False, history=3000, retrain=False, median_len=50, load=None, dataset='activity', quantization=0.0, max_len=-1, sample_rate=1.0, mask_rat

## Activity Extrapolation: Default Model Training (Seed 0)

### Objective
Following the quickstart and five-epoch trial, extend training to assess
whether the validation error stabilizes and establish a reference model
for subsequent baseline comparisons and ablation studies.

### Experimental Setup
- Dataset: Human Activity
- Task: Extrapolation (`forecasting`)
- Original-record split: 15 training, 5 validation, and 5 test records
- Model: ITSPM with the provided Activity preset
- Random seed: 0
- Maximum training epochs: 100
- Early stopping: Stop after 10 consecutive epochs without improvement
  in validation MSE
- Checkpoint selection: Lowest validation MSE
- Evaluation metrics: Test MSE and MAE

### Notes
Training starts from scratch rather than resuming the previous five-epoch run.
This is a single-seed experiment, not a complete three-seed reproduction.
Test results are reported for the checkpoint selected using validation MSE,
not used to select the model.

### Results
After training, record:
- Actual number of epochs completed
- Best epoch
- Best validation MSE
- Corresponding test MSE and MAE
- Total training time
- Checkpoint path

In [7]:
%cd /content/IP_ITSPM

!GPU=0 SEED=0 bash Extrapolation/scripts/activity.sh \
  --epoch 100 --patience 10

/content/IP_ITSPM
PID, device: 7649 cuda
/content/IP_ITSPM/regression.py
2026-10-23 07:21:09
regression.py --model ITSPM --gpu 0 --dataset activity --task forecasting --seed 0 --epoch 1000 --patience 10 --batch_size 32 --d_model 128 --dropout 0.05 --lr 5e-4 --weight_decay 1e-5 --n_ref_points 64 --n_scales 4 --n_mixer_layers 3 --history 3000 --state itspm_extrapolation_activity_best_seed0_20261023_072105 --epoch 100 --patience 10
Namespace(state='itspm_extrapolation_activity_best_seed0_20261023_072105', model='ITSPM', root_path='', data_path='/content/IP_ITSPM/data', n=100000000, epoch=100, batch_size=32, num_types=23, d_model=128, dropout=0.05, lr=0.0005, gpu='0', log='./logs/', save_path='./checkpoints/regression/', seed=0, patience=10, weight_decay=1e-05, load_path=None, test_only=False, logmode='a', task='forecasting', debug_flag=False, dp_flag=False, load_in_batch=False, history=3000, retrain=False, median_len=50, load=None, dataset='activity', quantization=0.0, max_len=-1, sample_

### Results
- Training stopped after 45 epochs.
- Early stopping was triggered after 10 consecutive epochs without
  improvement in validation MSE.
- The best checkpoint was selected at epoch 35
  (zero-based epoch index: 34).
- Best validation MSE: 0.0027030
- Corresponding test MSE: 0.0026396
- Corresponding test MAE: 0.03130
- Corresponding test RMSE: 0.05138

### Interpretation
Training loss continued to decrease gradually, but validation MSE did
not improve beyond epoch 35. The checkpoint from epoch 35 was therefore
retained rather than the model from the final epoch.

This establishes a single-seed ITSPM reference result on Human Activity.
Baseline comparisons and additional seeds are still required before
drawing conclusions about model performance.

## Baseline 1: Last-Value Persistence

### Objective
Evaluate a simple forecasting baseline before introducing additional
learned models or modifying ITSPM.

### Method
For each variable, use its most recent valid historical observation
as the prediction at every future query timestamp.

If a variable has no historical observation in a sample, use its mean
over observed training-history entries as a fallback.

### Evaluation Protocol
- Dataset: Human Activity
- Task: Forecasting
- History window: 3000 ms
- Forecast window: The repository's default Activity forecast window
- Data loading and normalization: Reuse the repository implementation
- Metrics: Reuse the repository's masked MSE and MAE evaluation
- No model training or access to future target values is required

### Results
Record validation and test MSE/MAE after execution.
Compare the test results with the validation-selected ITSPM checkpoint.

In [10]:
%cd /content/IP_ITSPM

import torch
import pandas as pd
import lib.utils as utils

from regression import parser
from lib.parse_datasets import parse_datasets
from lib.evaluation import evaluation

args = parser.parse_args([
    "--model", "ITSPM",
    "--dataset", "activity",
    "--task", "forecasting",
    "--seed", "0",
    "--history", "3000",
    "--batch_size", "32",
    "--d_model", "128",
    "--dropout", "0.05",
    "--lr", "0.0005",
    "--weight_decay", "0.00001",
    "--n_ref_points", "64",
    "--n_scales", "4",
    "--n_mixer_layers", "3",
    "--collate", "indseq",
])

args.device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

utils.setup_seed(args.seed)
data_obj = parse_datasets(args, length_stat=False)

print("Device:", args.device)
print("History window:", args.history)
print("Forecast window:", args.pred_window)
print("Number of variables:", data_obj["input_dim"])
print("Training batches:", data_obj["n_train_batches"])
print("Validation batches:", data_obj["n_val_batches"])
print("Test batches:", data_obj["n_test_batches"])

/content/IP_ITSPM
Total records: 25
Dataset n_samples: 25 15 5 5
Test record ids (first 20): ['B04', 'D02', 'A01', 'E04', 'C02']
Test record ids (last 20): ['B04', 'D02', 'A01', 'E04', 'C02']
data_max: tensor([5.6217, 3.9067, 2.4131, 5.3456, 3.4429, 2.2131, 5.7582, 3.9781, 2.6061,
        5.3913, 3.9654, 2.6003], device='cuda:0')
data_min: tensor([ 0.1852, -0.0871, -1.4615,  0.4127, -0.4760, -2.5436, -0.2787, -0.4522,
        -2.0269,  0.0211, -0.4482, -1.9588], device='cuda:0')
time_max: tensor(304861., device='cuda:0')
manual set time_max: tensor(4000)
Dataset n_samples after time split: 5400 3318 1007 1075
Device: cuda
History window: 3000
Forecast window: 1000
Number of variables: 12
Training batches: 104
Validation batches: 32
Test batches: 34


In [11]:
from pathlib import Path
import json
import subprocess

n_variables = data_obj["input_dim"]
value_sum = torch.zeros(n_variables, device=args.device)
value_count = torch.zeros(n_variables, device=args.device)

with torch.no_grad():
    for _ in range(data_obj["n_train_batches"]):
        batch = utils.get_next_batch(data_obj["train_dataloader"])
        values = batch["observed_data"]
        mask = batch["observed_mask"]

        value_sum += (values * mask).sum(dim=(0, 1))
        value_count += mask.sum(dim=(0, 1))

global_mean = value_sum.sum() / value_count.sum().clamp_min(1)

fallback_mean = torch.where(
    value_count > 0,
    value_sum / value_count.clamp_min(1),
    global_mean,
)


class LastValuePersistence(torch.nn.Module):
    def __init__(self, fallback):
        super().__init__()
        self.register_buffer("fallback", fallback)

    def forecasting(
        self, tp_to_predict, observed_data,
        observed_tp, observed_mask
    ):
        valid = observed_mask.bool()
        times = observed_tp

        if times.ndim == 1:
            times = times[None, :, None]
        elif times.ndim == 2:
            times = times[:, :, None]

        times = times.expand_as(observed_data)

        last_index = (
            times.masked_fill(~valid, float("-inf"))
            .argmax(dim=1, keepdim=True)
        )

        last_value = observed_data.gather(
            dim=1, index=last_index
        ).squeeze(1)

        has_history = valid.any(dim=1)
        last_value = torch.where(
            has_history,
            last_value,
            self.fallback[None, :],
        )

        query_length = tp_to_predict.shape[1]

        return (
            last_value[:, None, :]
            .expand(-1, query_length, -1)
            .unsqueeze(0)
        )


baseline = LastValuePersistence(fallback_mean).to(args.device)
baseline.eval()

with torch.no_grad():
    validation_metrics = evaluation(
        baseline,
        data_obj["val_dataloader"],
        data_obj["n_val_batches"],
    )

    test_metrics = evaluation(
        baseline,
        data_obj["test_dataloader"],
        data_obj["n_test_batches"],
    )

results = pd.DataFrame([
    {
        "Model": "Last-value persistence",
        "Split": split,
        "MSE": metrics["mse"],
        "MAE": metrics["mae"],
    }
    for split, metrics in [
        ("Validation", validation_metrics),
        ("Test", test_metrics),
    ]
])

output_dir = Path("outputs/activity_last_value_seed0")
output_dir.mkdir(parents=True, exist_ok=True)

results.to_csv(output_dir / "metrics.csv", index=False)

metadata = {
    "model": "Last-value persistence",
    "dataset": args.dataset,
    "seed": args.seed,
    "history": args.history,
    "forecast_window": args.pred_window,
    "fallback": "Mean of observed training-history entries per variable",
    "source_commit": subprocess.check_output(
        ["git", "rev-parse", "HEAD"], text=True
    ).strip(),
    "validation": validation_metrics,
    "test": test_metrics,
}

with open(output_dir / "metrics.json", "w") as f:
    json.dump(metadata, f, indent=2)

display(results)
print("Results saved to:", output_dir.resolve())

,Model,Split,MSE,MAE
0,Last-value persistence,Validation,0.003878,0.034822
1,Last-value persistence,Test,0.003686,0.034079


Results saved to: /content/IP_ITSPM/outputs/activity_last_value_seed0


### Preliminary Comparison with ITSPM

| Model | Validation MSE | Test MSE | Test MAE |
|---|---:|---:|---:|
| Last-value persistence | 0.003878 | 0.003686 | 0.034079 |
| ITSPM (seed 0) | 0.0027030 | 0.0026396 | 0.03130 |

For the current Activity experiment, ITSPM achieves approximately
28.4% lower test MSE and 8.2% lower test MAE than last-value persistence.

Both methods use a 3000 ms history window and a 1000 ms forecast
window, with matching data splits and repository evaluation functions.

These are preliminary single-seed results. Additional training seeds
are required to assess the stability of ITSPM's performance.
This comparison alone does not isolate the contribution of the
temporal kernel branch.

## Activity Reproduction: Additional Training Seeds

### Objective
Assess the stability of ITSPM by completing training seeds 1 and 2,
in addition to the completed seed 0 experiment.

### Protocol
- Keep the model, preprocessing, and observation/forecast windows unchanged.
- Use the Activity preset with a maximum of 1000 epochs.
- Apply early stopping with patience 10.
- Select checkpoints using validation MSE only.
- Report test MSE and MAE as mean ± sample standard deviation
  across seeds 0, 1, and 2.

The completed seed 0 run used a 100-epoch cap but stopped at epoch 45.
Its cap was therefore not reached; retain the run and document this
difference in the experiment settings.

In [18]:
%cd /content/IP_ITSPM

!PYTHONUNBUFFERED=1 GPU=0 SEED=1 bash Extrapolation/scripts/activity.sh \
  --epoch 1000 --patience 10

/content/IP_ITSPM
PID, device: 24447 cuda
/content/IP_ITSPM/regression.py
2026-10-23 08:27:33
regression.py --model ITSPM --gpu 0 --dataset activity --task forecasting --seed 1 --epoch 1000 --patience 10 --batch_size 32 --d_model 128 --dropout 0.05 --lr 5e-4 --weight_decay 1e-5 --n_ref_points 64 --n_scales 4 --n_mixer_layers 3 --history 3000 --state itspm_extrapolation_activity_best_seed1_20261023_082729 --epoch 1000 --patience 10
Namespace(state='itspm_extrapolation_activity_best_seed1_20261023_082729', model='ITSPM', root_path='', data_path='/content/IP_ITSPM/data', n=100000000, epoch=1000, batch_size=32, num_types=23, d_model=128, dropout=0.05, lr=0.0005, gpu='0', log='./logs/', save_path='./checkpoints/regression/', seed=1, patience=10, weight_decay=1e-05, load_path=None, test_only=False, logmode='a', task='forecasting', debug_flag=False, dp_flag=False, load_in_batch=False, history=3000, retrain=False, median_len=50, load=None, dataset='activity', quantization=0.0, max_len=-1, samp

In [19]:
from pathlib import Path
from datetime import datetime
import os
import shutil
import subprocess
import tarfile

repo = Path("/content/IP_ITSPM")
drive_root = Path("/content/drive/MyDrive")

assert os.path.ismount("/content/drive"), "Google Drive is not mounted."
assert repo.is_dir(), "Cannot find /content/IP_ITSPM."

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_dir = drive_root / "ARIN6900" / "snapshots" / stamp
backup_dir.mkdir(parents=True, exist_ok=False)

archive_local = Path(f"/content/ITSPM_snapshot_{stamp}.tar.gz")
archive_drive = backup_dir / archive_local.name

def exclude_files(info):
    excluded = {".git", "__pycache__", ".ipynb_checkpoints", ".pytest_cache"}
    return None if excluded.intersection(Path(info.name).parts) else info

with tarfile.open(archive_local, "w:gz") as tar:
    tar.add(repo, arcname="IP_ITSPM", filter=exclude_files)

shutil.copy2(archive_local, archive_drive)

(backup_dir / "source_commit.txt").write_text(
    subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=repo,
        text=True
    )
)

(backup_dir / "git_status.txt").write_text(
    subprocess.check_output(
        ["git", "status", "--short"],
        cwd=repo,
        text=True
    )
)

print("Saved to:", archive_drive)
print("Size (MB):", round(archive_drive.stat().st_size / 1024**2, 2))

Saved to: /content/drive/MyDrive/ARIN6900/snapshots/20261023_090130/ITSPM_snapshot_20261023_090130.tar.gz
Size (MB): 41.69
